# 03 — Resources, relationships, and knowledge levels

This notebook answers four questions about the **full Tier 0 graph**:

1. Which primary knowledge sources contribute edges, and how many does each contribute?
2. Which relationship types does each source provide?
3. Which knowledge levels occur, including missing values and `not_provided`?
4. How do knowledge levels vary by source and predicate?

Sections 1–2 use release metadata. Sections 3–4 use a cached streaming scan of raw
edges. There is no chemical–disease filter. Counts describe edge records, not unique
pairs, independent studies, or evidence quality. Aggregator sources are not included
in primary-source breakdowns.


In [1]:
from pathlib import Path
import csv
import json
from html import escape
from IPython.display import HTML, display

ROOT = next(p / "experiments" / "ultra" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "experiments" / "ultra" / "download.json").exists())
manifest = json.loads((ROOT / "download.json").read_text())
metadata_file = next(f["path"] for f in manifest["files"] if f["path"].endswith("graph-metadata.json"))
metadata = json.loads((ROOT / metadata_file).read_text())
assert metadata["version"] == manifest["release"]
summary = metadata["schema"]["edges_summary"]
source_counts = summary["primary_knowledge_sources"]
source_predicates = summary["predicates_by_knowledge_source"]
OUTPUT = ROOT / "output" / "resources"
OUTPUT.mkdir(parents=True, exist_ok=True)

# Internal consistency checks, not an independent raw-record recount.
assert set(source_counts) == set(source_predicates)
assert all(source_counts[s] == sum(source_predicates[s].values()) for s in source_counts)

def table(rows, columns):
    def cell(value):
        if isinstance(value, int):
            value = f"{value:,}"
        return escape(str(value)).replace("\n", "<br>")
    header = "".join(f"<th>{escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td style='text-align:left'>{cell(r[c])}</td>" for c in columns) + "</tr>" for r in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

def save_csv(filename, rows):
    with (OUTPUT / filename).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

print(f"Release: {metadata['version']}")
print(f"Reported graph edges: {summary['total_count']:,}")
print(f"Primary-source labels: {len(source_counts)}")
print("Metadata source counts and per-source predicate subtotals agree.")


Release: 2026_06_21
Reported graph edges: 28,709,074
Primary-source labels: 35
Metadata source counts and per-source predicate subtotals agree.


## 1. Which sources contribute edges, and how many?

Each row is one primary-source identifier (`infores:...`). **Share of graph edges**
is its reported edge count divided by the full graph's edge count. **Predicate types**
counts distinct relationship labels contributed by that source.

A larger contribution means more records, not necessarily stronger evidence.


In [2]:
source_rows = [
    {"Source": source, "Edge records": count,
     "Share of graph edges (%)": round(100 * count / summary["total_count"], 2),
     "Predicate types": len(source_predicates[source])}
    for source, count in sorted(source_counts.items(), key=lambda item: (-item[1], item[0]))
]
table(source_rows, ["Source", "Edge records", "Share of graph edges (%)", "Predicate types"])
save_csv("source_counts.csv", source_rows)
print(f"Sum of reported source counts: {sum(source_counts.values()):,}")
print("For this release, that sum equals the graph's edge count.")


Source,Edge records,Share of graph edges (%),Predicate types
infores:ubergraph,"10,120,364",35.25,1
infores:bgee,"4,654,943",16.21,1
infores:ctd,"4,197,763",14.62,6
infores:pathbank,"1,465,291",5.1,3
infores:text-mining-provider-targeted,"1,380,234",4.81,3
infores:semmeddb,"1,308,287",4.56,19
infores:bindingdb,"1,139,112",3.97,1
infores:goa,"1,043,681",3.64,13
infores:geneticskp,"891,558",3.11,3
infores:intact,"622,361",2.17,1


Sum of reported source counts: 28,709,074
For this release, that sum equals the graph's edge count.


## 2. Which relationships does each source provide?

The overview shows the **three most frequent predicates per source**, with their
edge counts. This helps distinguish resources contributing ontology relationships,
functional annotations, molecular interactions, or treatment-related records.
The complete source–predicate table is saved to CSV.


In [3]:
relationship_rows = []
overview_rows = []
for source in (row["Source"] for row in source_rows):
    predicates = sorted(source_predicates[source].items(), key=lambda item: (-item[1], item[0]))
    for predicate, count in predicates:
        relationship_rows.append({"Source": source, "Predicate": predicate, "Edge records": count,
                                  "Share of source edges (%)": round(100 * count / source_counts[source], 2)})
    overview_rows.append({"Source": source,
                          "Most frequent predicates (edge records)": "\n".join(
                              f"{p.removeprefix('biolink:')} ({n:,})" for p, n in predicates[:3]),
                          "Other predicate types": max(0, len(predicates) - 3)})
table(overview_rows, ["Source", "Most frequent predicates (edge records)", "Other predicate types"])
save_csv("source_predicates.csv", relationship_rows)


Source,Most frequent predicates (edge records),Other predicate types
infores:ubergraph,"subclass_of (10,120,364)",0
infores:bgee,"expressed_in (4,654,943)",0
infores:ctd,"associated_with (2,217,102)affects (1,869,701)correlated_with (67,537)",3
infores:pathbank,"occurs_in (801,048)has_participant (664,234)affects (9)",0
infores:text-mining-provider-targeted,"affects (968,665)treats_or_applied_or_studied_to_treat (300,900)contributes_to (110,669)",0
infores:semmeddb,"located_in (318,354)affects (209,703)coexists_with (133,647)",16
infores:bindingdb,"directly_physically_interacts_with (1,139,112)",0
infores:goa,"involved_in (382,583)enables (226,560)located_in (216,276)",10
infores:geneticskp,"genetic_association (415,530)condition_associated_with_gene (238,014)gene_associated_with_condition (238,014)",0
infores:intact,"physically_interacts_with (622,361)",0


Change `SOURCE` below to inspect **all predicates for one source**. The default
is GO annotations (`infores:goa`); try `infores:drugcentral`, `infores:ctd`, or
`infores:semmeddb`. Percentages below use that source's edge count as the denominator.


In [4]:
SOURCE = "infores:goa"
if SOURCE not in source_counts:
    raise ValueError("Choose a source identifier from section 1.")
print(f"{SOURCE}: {source_counts[SOURCE]:,} edge records")
table([row for row in relationship_rows if row["Source"] == SOURCE],
      ["Predicate", "Edge records", "Share of source edges (%)"])
print(f"\nFull tables saved in: {OUTPUT}")


infores:goa: 1,043,681 edge records


Predicate,Edge records,Share of source edges (%)
biolink:involved_in,"382,583",36.66
biolink:enables,"226,560",21.71
biolink:located_in,"216,276",20.72
biolink:acts_upstream_of_or_within,"93,606",8.97
biolink:active_in,"81,227",7.78
biolink:part_of,"35,120",3.37
biolink:contributes_to,"3,422",0.33
biolink:colocalizes_with,"2,023",0.19
biolink:acts_upstream_of,"1,983",0.19
biolink:acts_upstream_of_positive_effect,329,0.03



Full tables saved in: /Users/carolinagonzalez/shepherd/experiments/ultra/output/resources


## 3. Which knowledge levels occur in the graph?

`knowledge_level` describes the kind of claim: for example, an assertion, prediction,
observation, or statistical association. It is not a confidence score or an experimental
evidence code. See the [Biolink definitions](https://biolink.github.io/biolink-model/KnowledgeLevelEnum/).

The next cell runs [inspect_knowledge_levels.py](../scripts/inspect_knowledge_levels.py).
Its first run reads every raw edge without extracting the archive; later runs reuse
matching cached counts. No edges are filtered. Missing fields, nulls, empty strings,
and the explicit value `not_provided` are counted separately. Unrecognized values
are retained rather than silently classified.


In [5]:
import subprocess
import sys
from collections import Counter

subprocess.run([sys.executable, str(ROOT / "scripts" / "inspect_knowledge_levels.py")], check=True)
knowledge = json.loads((OUTPUT / "knowledge_levels.json").read_text())
assert knowledge["release"] == metadata["version"]
print(f"Raw edges scanned: {knowledge['edges_scanned']:,}")
for check, matches in knowledge["checks"].items():
    print(f"{check}: {matches}")
assert all(knowledge["checks"].values()), "Investigate discrepancies before comparing these tables."

expected_levels = ["knowledge_assertion", "logical_entailment", "prediction",
                   "statistical_association", "text_co_occurrence", "observation", "not_provided",
                   "[missing field]", "[null]", "[empty string]"]
level_counts = knowledge["levels"]
level_names = sorted(set(expected_levels) | set(level_counts), key=lambda level: (-level_counts.get(level, 0), level))
level_rows = [{"Knowledge level": level, "Edge records": level_counts.get(level, 0),
               "Share of graph edges (%)": round(100 * level_counts.get(level, 0) / knowledge["edges_scanned"], 2)}
              for level in level_names]
table(level_rows, ["Knowledge level", "Edge records", "Share of graph edges (%)"])
save_csv("knowledge_level_counts.csv", level_rows)


Using cached knowledge-level counts: /Users/carolinagonzalez/shepherd/experiments/ultra/output/resources/knowledge_levels.json
Raw edges scanned: 28,709,074
edge_total_matches_metadata: True
source_totals_match_metadata: True
source_predicate_totals_match_metadata: True


Knowledge level,Edge records,Share of graph edges (%)
knowledge_assertion,"21,878,156",76.21
statistical_association,"3,113,335",10.84
not_provided,"2,707,546",9.43
prediction,"788,724",2.75
logical_entailment,"156,059",0.54
observation,"65,254",0.23
[empty string],0,0.0
[missing field],0,0.0
[null],0,0.0
text_co_occurrence,0,0.0


## 4. How do knowledge levels vary by source and predicate?

First compare sources. Percentages are **within each source**, so a small resource
can be compared with a large one. Only observed levels are listed for each source.

An edge is counted once for each distinct primary source it names. If an edge names
multiple primary sources, source counts can overlap; global counts still count that
edge once. The scan reports how often that happens.


In [6]:
def knowledge_breakdown(rows, key):
    grouped = {}
    for row in rows:
        grouped.setdefault(row[key], Counter())[row["knowledge_level"]] += row["edges"]
    result = []
    for label, counts in sorted(grouped.items(), key=lambda item: (-sum(item[1].values()), item[0])):
        total = sum(counts.values())
        result.append({"Source": label, "Edge records": total,
                       "Knowledge levels (% within row)": "\n".join(
                           f"{level}: {count:,} ({100 * count / total:.2f}%)"
                           for level, count in counts.most_common())})
    return result

print(f"Edges without a primary source: {knowledge['edges_without_primary_source']:,}")
print(f"Edges with multiple primary sources: {knowledge['edges_with_multiple_primary_sources']:,}")
table(knowledge_breakdown(knowledge["by_source"], "source"),
      ["Source", "Edge records", "Knowledge levels (% within row)"])
save_csv("knowledge_levels_by_source.csv", knowledge["by_source"])
save_csv("knowledge_levels_by_predicate.csv", knowledge["by_predicate"])
save_csv("knowledge_levels_by_source_predicate.csv", knowledge["by_source_predicate"])


Edges without a primary source: 0
Edges with multiple primary sources: 0


Source,Edge records,Knowledge levels (% within row)
infores:ubergraph,"10,120,364","knowledge_assertion: 10,120,364 (100.00%)"
infores:bgee,"4,654,943","knowledge_assertion: 4,654,943 (100.00%)"
infores:ctd,"4,197,763","statistical_association: 2,221,693 (52.93%)knowledge_assertion: 1,976,070 (47.07%)"
infores:pathbank,"1,465,291","knowledge_assertion: 1,465,291 (100.00%)"
infores:text-mining-provider-targeted,"1,380,234","not_provided: 1,380,234 (100.00%)"
infores:semmeddb,"1,308,287","not_provided: 1,308,287 (100.00%)"
infores:bindingdb,"1,139,112","knowledge_assertion: 1,139,112 (100.00%)"
infores:goa,"1,043,681","prediction: 788,724 (75.57%)knowledge_assertion: 235,932 (22.61%)not_provided: 19,025 (1.82%)"
infores:geneticskp,"891,558","statistical_association: 891,558 (100.00%)"
infores:intact,"622,361","knowledge_assertion: 622,361 (100.00%)"


Select a predicate below to compare the sources contributing that particular
relationship. Start with `biolink:treats`; try `biolink:associated_with`,
`biolink:involved_in`, or `biolink:affects`.

This includes **all endpoint categories**, unlike notebook 02's filtered
chemical–disease pair counts. Percentages below are within each source–predicate group.


In [7]:
PREDICATE = "biolink:treats"
selected = [row for row in knowledge["by_source_predicate"] if row["predicate"] == PREDICATE]
if not selected:
    raise ValueError("Choose a predicate present in section 2's tables.")
print(PREDICATE)
table(knowledge_breakdown(selected, "source"),
      ["Source", "Edge records", "Knowledge levels (% within row)"])
print(f"\nComplete counts for every source–predicate group are saved in {OUTPUT}.")


biolink:treats


Source,Edge records,Knowledge levels (% within row)
infores:multiomics-clinicaltrials,"12,906","knowledge_assertion: 12,906 (100.00%)"
infores:drugcentral,"9,608","knowledge_assertion: 9,608 (100.00%)"
infores:multiomics-drugapprovals,"5,095","knowledge_assertion: 5,095 (100.00%)"
infores:drug-repurposing-hub,"3,572","knowledge_assertion: 3,572 (100.00%)"
infores:ttd,"2,453","knowledge_assertion: 2,453 (100.00%)"



Complete counts for every source–predicate group are saved in /Users/carolinagonzalez/shepherd/experiments/ultra/output/resources.


These tables describe the labels recorded by the graph's providers. They do not
verify the claims or identify the best evidence for ULTRA. `knowledge_assertion`
does not automatically mean experimental support, and `not_provided` does not
mean a relationship is false. A pair can have edges with different knowledge levels.

We have not yet inspected representative records or evaluated any evidence filter.
